# 01 — Sephora Dataset: Initial EDA

Exploratory profiling of the **Sephora Products & Skincare Reviews** dataset.

- `product_info.csv` — product catalogue
- `reviews_*.csv` — ~1M reviews (read in chunks to stay memory-safe)

Run cells top-to-bottom.

In [ ]:
import os, glob
import pandas as pd
import numpy as np

# make sure we run from the project root (works if notebook is in notebooks/)
if os.path.basename(os.getcwd()).lower() == 'notebooks':
    os.chdir('..')
print('cwd:', os.getcwd())

DATA = os.path.join('data', 'sephora', 'archive')
assert os.path.isdir(DATA), f'Data folder not found: {DATA}'
print('files:', os.listdir(DATA))

## 1. Products (`product_info.csv`)

In [ ]:
prod = pd.read_csv(os.path.join(DATA, 'product_info.csv'))
print(f'products: {len(prod):,}  |  columns: {prod.shape[1]}  |  brands: {prod["brand_name"].nunique():,}')
prod.head(3)

In [ ]:
# missingness for key columns
key_cols = ['rating','reviews','ingredients','price_usd','primary_category',
            'secondary_category','tertiary_category','highlights','size']
miss = prod[key_cols].isna().sum().to_frame('missing')
miss['pct'] = (100*miss['missing']/len(prod)).round(1)
miss

In [ ]:
# price + rating summary
print('PRICE (USD)')
print(prod['price_usd'].describe(percentiles=[.25,.5,.75,.95]).round(2))
print('\nRATING (0-5)')
print(prod['rating'].describe().round(2))

In [ ]:
# category breakdown + skincare focus
display(prod['primary_category'].value_counts().head(12).to_frame('products'))
skin_mask = prod['primary_category'].astype(str).str.contains('Skincare', case=False, na=False)
print(f'Skincare products: {skin_mask.sum():,} ({100*skin_mask.sum()/len(prod):.1f}%)')

## 2. Reviews (`reviews_*.csv`) — chunked aggregation

~1M rows across 5 files. We stream them in 100k-row chunks and accumulate counts.

In [ ]:
review_files = sorted(glob.glob(os.path.join(DATA, 'reviews_*.csv')))
usecols = ['author_id','rating','is_recommended','review_text','skin_tone','skin_type','product_id']

total = 0
rating_counts = pd.Series(dtype='int64')
skin_type_counts = pd.Series(dtype='int64')
skin_tone_counts = pd.Series(dtype='int64')
missing_text = missing_skin_type = missing_skin_tone = 0
text_len_sum = text_len_n = 0
unique_products, unique_authors = set(), set()

for f in review_files:
    for ch in pd.read_csv(f, usecols=usecols, chunksize=100_000, low_memory=False):
        total += len(ch)
        rating_counts = rating_counts.add(ch['rating'].value_counts(), fill_value=0)
        skin_type_counts = skin_type_counts.add(ch['skin_type'].value_counts(), fill_value=0)
        skin_tone_counts = skin_tone_counts.add(ch['skin_tone'].value_counts(), fill_value=0)
        missing_text += ch['review_text'].isna().sum()
        missing_skin_type += ch['skin_type'].isna().sum()
        missing_skin_tone += ch['skin_tone'].isna().sum()
        txt = ch['review_text'].dropna().astype(str)
        text_len_sum += txt.str.len().sum(); text_len_n += len(txt)
        unique_products.update(ch['product_id'].dropna().unique())
        unique_authors.update(ch['author_id'].dropna().unique())
    print('done:', os.path.basename(f))

print(f'\nTotal reviews: {total:,}')
print(f'Unique products reviewed: {len(unique_products):,}')
print(f'Unique reviewers: {len(unique_authors):,}')
print(f'Missing review_text: {missing_text:,} ({100*missing_text/total:.1f}%)')
print(f'Avg review length: {text_len_sum/text_len_n:.0f} chars')

In [ ]:
# rating distribution
rd = rating_counts.sort_index().astype(int).to_frame('count')
rd['pct'] = (100*rd['count']/total).round(1)
rd

In [ ]:
# reviewer skin_type
st = skin_type_counts.sort_values(ascending=False).astype(int).to_frame('count')
st['pct'] = (100*st['count']/total).round(1)
st.loc['(missing)'] = [missing_skin_type, round(100*missing_skin_type/total,1)]
st

In [ ]:
# reviewer skin_tone  --  KEY for the fairness angle
sn = skin_tone_counts.sort_values(ascending=False).astype(int).to_frame('count')
sn['pct'] = (100*sn['count']/total).round(1)
sn.loc['(missing)'] = [missing_skin_tone, round(100*missing_skin_tone/total,1)]
sn

### Takeaways
- Ratings are heavily skewed to 5★ → class imbalance to handle in sentiment modelling.
- `skin_type` is ~90% populated and clean → primary recommender filter.
- `skin_tone` is severely imbalanced (light/fair dominate, deep/dark tiny) → this is the **representation-bias evidence** for the fairness contribution.

Next notebook: `02_clean_and_build.ipynb` builds the recommender data layer + charts.